# 📋 Verificador de Submissões e Editais (IFC - Campus Araquari)
### Leitura automatizada a partir da pasta montada no Google Drive

Este notebook lê diretamente a pasta do Google Drive contendo as subpastas dos projetos (`PVG`), realiza o diagnóstico de documentos faltantes, analisa o mérito curricular (Anexo II) e permite inspecionar cada submissão individualmente com exportação em CSV.

## 1. Conectar ao Google Drive
Execute a célula abaixo para autorizar o acesso à sua conta e montar o Drive no Colab:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalação das bibliotecas necessárias

In [ ]:
!pip install -q pymupdf python-docx pandas

## 3. Configuração do Caminho da Pasta no Drive
Indique abaixo o caminho da pasta onde estão as subpastas `PVG...`:

In [ ]:
import os
from pathlib import Path

# Ajuste o caminho conforme a localização da pasta no seu Google Drive:
PASTA_DRIVE = Path("/content/drive/MyDrive/SuaPastaDosEditais")

if not PASTA_DRIVE.exists():
    print(f"⚠️ Atenção: Caminho não encontrado: {PASTA_DRIVE}")
    print("Dica: Busque pelo nome da sua pasta no Drive usando o comando abaixo:")
    !find /content/drive/MyDrive -maxdepth 3 -type d 2>/dev/null
else:
    subpastas = [d.name for d in PASTA_DRIVE.iterdir() if d.is_dir()]
    print(f"✅ Pasta localizada com sucesso! Total de subpastas encontradas: {len(subpastas)}")

## 4. Importar o Módulo de Análise e Executar o Diagnóstico

In [ ]:
from src.analise_editais import analisar_todos_editais, gerar_dataframes, gerar_dataframe_submissao

projetos = analisar_todos_editais(PASTA_DRIVE)
print(f"✅ {len(projetos)} editais/submissões analisados com sucesso!")

dfs = gerar_dataframes(projetos)
df_resumo = dfs['resumo']
display(df_resumo[['Código', 'Coordenador', 'Possui Tarja', 'Possui Mérito (Anexo II)', 'Pontos Mérito', 'Status Conformidade']])

## 5. Inspeção Detalhada por Submissão (Interativo)
Escolha uma submissão para ver os detalhes, checklist de documentos faltantes, equipe e pontuação de mérito:

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

mapa_proj = {f"{p.codigo} - {p.titulo[:50]}... ({p.coordenador})": p for p in projetos}

dropdown = widgets.Dropdown(
    options=list(mapa_proj.keys()),
    description='Edital/Projeto:',
    layout={'width': '80%'}
)

def exibir_detalhes(escolha):
    p = mapa_proj[escolha]
    print('=' * 80)
    print(f"{p.codigo}: {p.titulo}")
    print(f"Coordenador: {p.coordenador} | Edital: {p.edital} | Área: {p.area}")
    print(f"Pontuação Declarada no Mérito: {p.pontuacao_merito} pontos")
    print('=' * 80)
    
    if not p.documentos_faltantes:
        print("🎉 TUDO CERTO! Todos os documentos obrigatórios estão presentes.")
    else:
        print(f"⚠️ DOCUMENTAÇÃO INCOMPLETA! Faltam {len(p.documentos_faltantes)} documento(s):")
        for doc in p.documentos_faltantes:
            print(f"  ❌ {doc}")
            
    print('\n📂 Arquivos Anexados:')
    for a in p.arquivos:
        print(f"  • {a.nome} -> {a.descricao} ({a.tamanho_kb} KB, {a.num_paginas} pág.)")
        
    if p.membros:
        print('\n👥 Equipe:')
        for m in p.membros:
            print(f"  • {m.nome} | {m.categoria} | CH: {m.ch_dedicada} | Papel: {m.tipo_participacao}")

widgets.interactive(exibir_detalhes, escolha=dropdown)

## 6. Exportar Relatório em CSV
Baixe a planilha consolidada com todas as submissões analisadas:

In [ ]:
from google.colab import files

caminho_csv = 'resumo_submissoes_editais.csv'
df_resumo.to_csv(caminho_csv, index=False, encoding='utf-8-sig')
print(f"✅ Planilha salva em: {caminho_csv}")
files.download(caminho_csv)